## GRPO Training on GSM8K with Together API

This notebook is a **step-by-step onboarding guide** for running **Group Relative Policy Optimization (GRPO)** on the [GSM8K](https://huggingface.co/datasets/openai/gsm8k) math dataset using the Together Post Training API.

**What is GRPO?**

GRPO is a reinforcement-learning algorithm for fine-tuning language models without a separate value network. For each prompt it:

1. Samples a **group** of completions from the current policy.
2. Scores every completion with a **reward function** (here: exact-match on the numerical answer inside `\\boxed{}`).
3. Computes per-sample **advantages** relative to the group mean reward.
4. Runs a forward-backward pass with the **GRPO loss**, updates the parameters, and publishes the new weights for sampling.

**Pipeline overview:**

```
Create session ──▶ [ Sample ──▶ Score ──▶ Forward/Backward ──▶ Optim Step ──▶ Weights Sync ] × N ──▶ Stop session
```

Training metrics are logged to **Weights & Biases** for live monitoring.

Prerequisites

Before running this notebook make sure you have:

1. Installed the `together` package (`uv sync` in this repo; the post-training helpers live in `together.post_training`).
2. Installed the `post_training_cookbook` package from this repo (`pip install -e .`).
3. Set `TOGETHER_API_KEY` and `HF_TOKEN` in a `.env` file at the repo root (or as environment variables).

In [ ]:
from __future__ import annotations

import os
import time

import datasets
import numpy as np
import wandb
from dotenv import load_dotenv
from together import AsyncTogether
from together.post_training import (
    AdamParams,
    ModelResourcesClient,
    Sample,
    SamplingParams,
    download_checkpoint_async,
)
from transformers import AutoTokenizer, PreTrainedTokenizerBase

from post_training_cookbook.datasets.registry import DATASETS, PromptExample
from post_training_cookbook.grpo.config import GRPOConfig, policy_loss
from post_training_cookbook.grpo.samples import build_policy_sample
from post_training_cookbook.samples import is_trainable, make_prompt

load_dotenv(override=True)

Configuring the Training Run

Key parameters explained:

| Parameter | Value | Description |
| --- | --- | --- |
| `MODEL` | `Qwen/Qwen3.8-27B` | Base model to fine-tune. |
| `BATCH_SIZE` | 4 | Number of prompts per training step. |
| `GROUP_SIZE` | 8 | Completions sampled per prompt (GRPO group size). |
| `MAX_STEPS` | 50 | Total training steps. |
| `MAX_PROMPT_LENGTH` | 512 | Max tokens for the prompt (few-shot + question). Defaults to `MAX_MODEL_LENGTH - MAX_SAMPLE_TOKENS` when using `GRPOConfig`. |
| `MAX_MODEL_LENGTH` | 2048 | Max total sequence length (prompt + response). The primary length knob. |
| `MAX_SAMPLE_TOKENS` | 1024 | Max new tokens the model may generate per sample. Defaults to `MAX_MODEL_LENGTH` when using `GRPOConfig`. |
| `LEARNING_RATE` | 1e-5 | Adam learning rate (LoRA RL ≈ 10× the full-weight RL LR of 1e-6). |
| `PRINT_EXAMPLES` | True | Print a decoded Question / Response each step. |

`DS_CONFIG` is loaded from the `post_training_cookbook` dataset registry, which defines how to format prompts, extract targets, compute rewards, and provides a few-shot example.

### Supported loss types

The cookbook GRPO loop names the server objective with `GRPOConfig.loss` (`"grpo"`, `"ppo"`, `"cispo"`, `"dppo"`, `"dro"`, `"importance_sampling"`). `policy_loss(config)` translates those knobs into the SDK `LossConfig`. Cross-entropy stays on the SFT path (`sft_example.ipynb`).

This notebook uses `policy_loss(GRPOConfig())` — today's DR-GRPO payload. To try another objective, pass knobs on the config (for example `GRPOConfig(loss="ppo")`). PPO reduces by token-sum rather than GRPO's fixed horizon, so it is not a drop-in.


In [ ]:
BASE_URL = os.environ.get("TOGETHER_BASE_URL", "https://api.together.ai/v1")
API_KEY = os.environ.get("TOGETHER_API_KEY", "")
MODEL = "Qwen/Qwen3.8-27B"
TOKENIZER_MODEL = MODEL
HF_TOKEN = os.environ.get("HF_TOKEN", "")

DS_CONFIG = DATASETS["gsm8k"]

BATCH_SIZE = 4
GROUP_SIZE = 8
MAX_STEPS = 50
MAX_PROMPT_LENGTH = 512
MAX_MODEL_LENGTH = 2048
MAX_SAMPLE_TOKENS = 1024
LEARNING_RATE = 1e-5
BETA1 = 0.9
BETA2 = 0.95
EPS = 1e-8
WEIGHT_DECAY = 0.1
TIMEOUT = 7200.0
PRINT_EXAMPLES = True
# Today's DR-GRPO payload. Pass knobs on GRPOConfig (e.g. loss="ppo") to change objective.
LOSS = policy_loss(GRPOConfig())

WANDB_PROJECT = "grpo-demo"
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", None)

Helper Functions

Most utility functions are imported from the `post_training_cookbook` library: reward functions from `post_training_cookbook.datasets`, GRPO sample building (`build_policy_sample`) from `post_training_cookbook.grpo.samples`, and prompt payload building (`make_prompt`) from `post_training_cookbook.samples`. The cell below defines two helpers used within this notebook:

| Function | Purpose |
| --- | --- |
| `build_prompt` | Formats a dataset row into chat messages (with few-shot examples), applies the tokenizer's chat template, and returns the prompt token IDs alongside the extraction target. |
| `score_completions` | Decodes sampled sequences with the tokenizer, computes rewards using the dataset's `compute_reward`, and returns them alongside token/logprob data. |

In [ ]:
def build_prompt(row: dict, *, tokenizer: PreTrainedTokenizerBase) -> PromptExample:
    """Format a dataset row into chat-templated prompt tokens."""
    question = DS_CONFIG.format_question(row)
    target = DS_CONFIG.extract_target(row)
    messages = list(DS_CONFIG.few_shot_messages) + [
        {"role": "user", "content": question},
    ]
    prompt_tokens = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)["input_ids"]
    if len(prompt_tokens) > MAX_PROMPT_LENGTH:
        raise ValueError(f"Prompt length {len(prompt_tokens)} exceeds {MAX_PROMPT_LENGTH}")
    return PromptExample(question=question, target=target, prompt_tokens=prompt_tokens)


def score_completions(
    sequences: list,
    *,
    tokenizer: PreTrainedTokenizerBase,
    target: str,
) -> tuple[list[float], list[tuple[list[int], list[float], float]], list[int]]:
    """Decode sampled sequences, compute rewards, return (rewards, sequence_data, lengths)."""
    prompt_rewards: list[float] = []
    prompt_sequences: list[tuple[list[int], list[float], float]] = []
    response_lengths: list[int] = []
    for seq in sequences:
        seq_tokens = [int(token) for token in (seq.tokens or [])]
        seq_logprobs = [float(value) for value in (seq.logprobs or [])]
        decoded = tokenizer.decode(seq_tokens, skip_special_tokens=True)
        reward = DS_CONFIG.compute_reward(decoded, target)
        prompt_rewards.append(reward)
        prompt_sequences.append((seq_tokens, seq_logprobs, reward))
        response_lengths.append(len(seq_tokens))
    return prompt_rewards, prompt_sequences, response_lengths

Loading the Dataset

[GSM8K](https://huggingface.co/datasets/openai/gsm8k) is a dataset of ~7.5 k grade-school math word problems with step-by-step solutions. Each example has a `question` and an `answer` field; the final numerical answer follows a `####` marker.

We use the HuggingFace tokenizer's built-in `apply_chat_template` to convert conversation messages into token IDs. This handles the model's chat format (special tokens, role markers, etc.) so prompts are encoded exactly as the model expects.

In [ ]:
if DS_CONFIG.hf_config:
    raw_dataset = datasets.load_dataset(DS_CONFIG.hf_name, DS_CONFIG.hf_config)
else:
    raw_dataset = datasets.load_dataset(DS_CONFIG.hf_name)

dataset = raw_dataset[DS_CONFIG.hf_split].shuffle(seed=0)

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_MODEL,
    use_fast=True,
    token=HF_TOKEN or None,
)

total_batches = len(dataset) // BATCH_SIZE
total_steps = min(MAX_STEPS, total_batches)

print(f"Dataset size: {len(dataset)} examples")
print(f"Training for {total_steps} steps ({BATCH_SIZE} prompts x {GROUP_SIZE} samples per step)")

## Creating a Training Session

Together separates the GPU pods that host the base model (a **model resource**) from the **training session** that runs on them, so starting a run is two steps:

1. `ModelResourcesClient.create` provisions the GPUs and loads the base model, polling until the resource reaches `MODEL_RESOURCES_STATUS_READY`.
2. `model_resources.create_session` attaches a training session to that resource.

This typically takes 3-5 minutes. Once attached, the session exposes four operations: **`session.generator.sample`**, **`session.trainer.forward_backward`**, **`session.trainer.optim_step`**, and **`session.trainer.weights_sync`**.

> **Note:** Jupyter notebooks run their own asyncio event loop, so we use the `_async` variants (`create_async`, `create_session_async`, `sample_async`, etc.) with `await` throughout this notebook. If you're running a standalone Python script instead, you can use the sync methods directly (e.g. `ModelResourcesClient.create(...)`, `session.generator.sample(...)`) — no `await` needed.

> **Tip:** `ModelResourcesClient.create` can be used as a context manager to automatically stop the resource (and its sessions) when you're done:
> ```python
> with ModelResourcesClient.create(base_model=MODEL, api_key=API_KEY) as model_resources:
>     session = model_resources.create_session()
>     result = session.generator.sample(prompt)
>     ...
> # resource is stopped and GPUs released on exit
> ```

In [ ]:
model_resources = await ModelResourcesClient.create_async(
    base_model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
)
session = await model_resources.create_session_async()

print(f"Session ready: {session.session_id}")

## Training Loop

Each training step performs the following sequence:

**1. Sample completions** — For every prompt in the batch, we ask the model to generate `GROUP_SIZE` independent completions via `session.generator.sample_batch()`.

**2. Score & compute advantages** — Each completion is decoded and scored with `DS_CONFIG.compute_reward` (exact-match on the extracted answer). The per-sample advantage is `reward − group_mean`, so completions better than the group average get positive advantage and vice-versa. This is the core GRPO idea: the group itself serves as the baseline.

**3. Build training samples** — Prompt tokens, response tokens, log-probs (from the generator), and advantages are packaged into `Sample` objects via `build_policy_sample` from the library. Every array goes in unshifted, in token space; the builder trims them all together onto target-aligned slots.

**4. Forward-backward** — `session.trainer.forward_backward()` computes the GRPO loss and back-propagates gradients on the server side.

**5. Optimizer step** — `session.trainer.optim_step()` applies AdamW to update the model parameters.

**6. Weights sync** — `session.trainer.weights_sync()` publishes the updated weights so subsequent sampling uses the new policy.

Metrics (`loss`, `accuracy`, `reward_std`, `mean_response_length`) are logged to Weights & Biases each step. When `PRINT_EXAMPLES` is enabled, a sample Question / Response pair is also printed for qualitative inspection.

In [ ]:
overall_start = time.monotonic()
with wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    config={
        "model": MODEL,
        "batch_size": BATCH_SIZE,
        "group_size": GROUP_SIZE,
        "max_steps": MAX_STEPS,
        "learning_rate": LEARNING_RATE,
    },
):
    for step in range(total_steps):
        batch_start = step * BATCH_SIZE
        batch_end = min((step + 1) * BATCH_SIZE, len(dataset))
        batch_rows = dataset.select(range(batch_start, batch_end))
        samples: list[Sample] = []
        rewards: list[float] = []
        all_response_lengths: list[int] = []
        step_example: tuple[str, str, float] | None = None

        # Build all prompts for the batch
        prompt_examples = [build_prompt(row, tokenizer=tokenizer) for row in batch_rows]
        prompts = [make_prompt(pe.prompt_tokens) for pe in prompt_examples]

        sample_results = await session.generator.sample_batch_async(
            prompts,
            num_samples=GROUP_SIZE,
            sampling_params=SamplingParams(max_tokens=MAX_SAMPLE_TOKENS),
            timeout=TIMEOUT,
        )

        for prompt_example, sample_result, row in zip(prompt_examples, sample_results, batch_rows):
            sequences = sample_result.sequences
            if not sequences:
                raise RuntimeError("Sample operation returned no sequences")

            target = prompt_example.target
            prompt_tokens = prompt_example.prompt_tokens

            # Score completions
            prompt_rewards, prompt_sequences, resp_lens = score_completions(
                sequences,
                tokenizer=tokenizer,
                target=target,
            )
            all_response_lengths.extend(resp_lens)

            if PRINT_EXAMPLES and step_example is None:
                question = row.get("question") or row.get("prompt") or str(row)
                best_idx = max(range(len(prompt_rewards)), key=lambda i: prompt_rewards[i])
                best_tokens = prompt_sequences[best_idx][0]
                step_example = (
                    question,
                    tokenizer.decode(best_tokens, skip_special_tokens=True),
                    prompt_rewards[best_idx],
                )

            # Compute advantages and build GRPO samples
            baseline = np.mean(prompt_rewards)
            for seq_tokens, seq_logprobs, reward in prompt_sequences:
                advantage = reward - baseline
                prompt_len, response_len = len(prompt_tokens), len(seq_tokens)
                loss_mask = [0] * prompt_len + [1] * response_len
                if not is_trainable(loss_mask):
                    # A completion that sampled nothing trains no slot, and the
                    # builder rejects that payload. Skip it rather than let the
                    # raise abort the step mid-run — this session holds GPUs.
                    continue
                samples.append(
                    build_policy_sample(
                        tokens=prompt_tokens + seq_tokens,
                        logprobs=[0.0] * prompt_len + seq_logprobs,
                        loss_mask=loss_mask,
                        advantage=advantage,
                    )
                )
                rewards.append(reward)

        if len(samples) < 8:
            raise RuntimeError(f"Need at least 8 samples for forward_backward, got {len(samples)}")

        # Forward-backward
        fb_result = await session.trainer.forward_backward_async(
            loss=LOSS,
            samples=samples,
            timeout=TIMEOUT,
        )
        loss_value = fb_result.loss

        await session.trainer.optim_step_async(
            adam_params=AdamParams(
                beta1=BETA1,
                beta2=BETA2,
                eps=EPS,
                weight_decay=WEIGHT_DECAY,
                learning_rate=LEARNING_RATE,
            ),
            timeout=TIMEOUT,
        )
        await session.trainer.weights_sync_async(
            weight_sync_type="WEIGHT_SYNC_TYPE_SYNCHRONOUS",
            timeout=TIMEOUT,
        )

        elapsed = time.monotonic() - overall_start
        reward_mean = float(np.mean(rewards))
        reward_std = float(np.std(rewards))
        mean_resp_len = float(np.mean(all_response_lengths))

        wandb.log(
            {
                "loss": loss_value,
                "accuracy": reward_mean,
                "reward_std": reward_std,
                "mean_response_length": mean_resp_len,
            },
            step=step,
        )

        print(f"[step {step}] loss={loss_value:.4f} accuracy={reward_mean:.3f} elapsed={elapsed:.1f}s")

        if PRINT_EXAMPLES and step_example is not None:
            q, resp, rw = step_example
            print(f"  Question: {q}")
            print(f"  Response: {resp}")
            print(f"  Reward:   {rw}")

## Saving & Retrieving Training Artifacts

After training, you have several options for saving and retrieving your model:

| Operation | What it does | When to use |
| --- | --- | --- |
| **Inference checkpoint** | Saves LoRA adapter, registers as a model you can download or serve | You want to use the trained model |
| **Training checkpoint** | Saves adapter + optimizer state + step | You want to pause and resume training later |
| **Download checkpoint** | Downloads model files to your local machine | You want local weights (merged or adapter-only) |
| **Session query** | Lists all checkpoints saved during the session | You want to review what was saved |

In [ ]:
# -- Inference checkpoint: save the trained model for serving or download --
print("Creating inference checkpoint...")
checkpoint = await session.create_inference_checkpoint_async()
print(f"Inference checkpoint created: {checkpoint.id}")

### Deploy the checkpoint

The inference checkpoint registered its weights in the model registry. Print the ID to deploy:


In [ ]:
reg = (await session.retrieve_async()).inference_checkpoints[-1].registration
print("adapter:", reg.adapter.id if reg and reg.adapter else None)
print("merged model:", reg.model.id if reg and reg.model else None)

Then deploy it on Dedicated Model Inference (together-py 3.0+ CLI):

```bash
# Production: replicas merge the adapter into the base model at startup
tg beta endpoints deploy <adapter_id> --endpoint my-model --merge --config <config_id> --project <project_id>

# Comparison: attach several adapters to one multi-LoRA deployment
tg beta endpoints deploy <adapter_id> --endpoint my-sweep --attach-adapter
```

Guides: [Deploy a checkpoint](https://docs.together.ai/docs/post-training/deployment), [Multi-LoRA deployment](https://docs.together.ai/docs/post-training/multi-lora-deployment), [Prepare for FP4 inference](https://docs.together.ai/docs/post-training/prepare-for-fp4-inference).

In [ ]:
# -- Download inference checkpoint to your local machine --
session_info = await session.retrieve_async()
latest_ckpt = session_info.inference_checkpoints[-1]
client = AsyncTogether()
print(f"Downloading checkpoint from step {latest_ckpt.step} ({latest_ckpt.id})...")

In [ ]:
# Option A: merged model (base weights + LoRA merged together)
merged_paths = await download_checkpoint_async(
    client,
    checkpoint_id=latest_ckpt.id,
    variant="CHECKPOINT_VARIANT_MERGED",
    output_dir="./my_model_merged",
)
print(f"Merged model: {len(merged_paths)} files → ./my_model_merged/")

In [ ]:
# Option B: adapter only
adapter_paths = await download_checkpoint_async(
    client,
    checkpoint_id=latest_ckpt.id,
    variant="CHECKPOINT_VARIANT_ADAPTER",
    output_dir="./my_model_adapter",
)
print(f"Adapter only: {len(adapter_paths)} files → ./my_model_adapter/")

In [ ]:
# -- Training checkpoint: save optimizer state so you can resume later --
training_ckpt = await session.create_training_checkpoint_async()
print(f"Training checkpoint: {training_ckpt.id}")
print(f'Resume by passing resume_from_checkpoint_id="{training_ckpt.id}" when re-creating the session')
# load_optimizer=False starts from those weights with a fresh optimizer (e.g. after SFT).

In [ ]:
# -- Session query: review all checkpoints saved during training --
# (Reuses the metadata fetched above; call session.retrieve_async() if running standalone.)
print(f"Session {session_info.id} — {session_info.status}")
print(f"Model resource: {session_info.resources_id}")
print(f"Current step: {session_info.step}")
print(f"\nInference checkpoints ({len(session_info.inference_checkpoints)}):")
for ckpt in session_info.inference_checkpoints:
    print(f"  Step {ckpt.step}: id={ckpt.id} registration={ckpt.registration}")
print(f"\nTraining checkpoints ({len(session_info.training_checkpoints)}):")
for ckpt in session_info.training_checkpoints:
    print(f"  Step {ckpt.step}: id={ckpt.id}")

## Async multi-turn GRPO

For multi-turn environments, `train_multi_turn` can overlap rollout collection with training. Set `GRPOConfig(max_steps_off_policy=1, ...)` to accept groups scheduled at most one optimizer step ago; `None` (the default) keeps the synchronous loop, and single-turn training keeps its existing loop. `groups_in_flight` controls the rollout workers and defaults to `batch_size`.

`num_generator_replicas` raises sampling capacity when provisioning, and `weight_sync_type` chooses how new weights reach the generator. See [Asynchronous multi-turn collection](https://docs.together.ai/docs/post-training/grpo#async-training) for the staleness budget, the drop accounting, and how the two settings interact.

Try [the tool-use game async recipe](../recipes/grpo/tool_use_game_async.py) for a 50-step run. Watch `async/train_wait_s` and `async/staleness_max` to see whether the overlap is paying off.


## Cleanup

Stop the model resource to release the GPUs. Stopping the resource also ends the attached training session.

In [ ]:
await session.detach_async()
await model_resources.stop_async()
print(f"Resource {model_resources.model_resources_id} stopped.")